# Retrain One Pareto-Optimal CNN and Run Inference

This notebook: 
1. Loads one Pareto-optimal point from optimization outputs.
2. Rebuilds the exact CNN and hyperparameters.
3. Retrains the model.
4. Evaluates on non-training data (validation + test).
5. Shows sample predictions and a classification report.

Expected files from previous run:
- `outputs/pareto_selected.csv` (preferred) or `outputs/pareto_front.csv`

In [ ]:
import os
import sys
from pathlib import Path

def locate_project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / 'scripts' / 'run_pipeline.sh').exists() and (cwd / 'src').exists():
        return cwd

    candidate = Path('/kaggle/working/MOML Project')
    if (candidate / 'scripts' / 'run_pipeline.sh').exists() and (candidate / 'src').exists():
        return candidate

    input_root = Path('/kaggle/input')
    if input_root.exists():
        for req in input_root.rglob('requirements.txt'):
            root = req.parent
            if (root / 'scripts' / 'run_pipeline.sh').exists() and (root / 'src').exists():
                return root

    raise FileNotFoundError('Project root not found. Please run this notebook from the project folder or Kaggle working copy.')

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Python path configured.')

In [ ]:
import pandas as pd

pareto_candidates = [
    PROJECT_ROOT / 'outputs' / 'pareto_selected.csv',
    PROJECT_ROOT / 'outputs' / 'pareto_front.csv',
]

pareto_csv = None
for p in pareto_candidates:
    if p.exists():
        pareto_csv = p
        break

if pareto_csv is None:
    raise FileNotFoundError('No Pareto CSV found. Run the optimization pipeline first to create outputs/pareto_selected.csv or outputs/pareto_front.csv.')

pareto_df = pd.read_csv(pareto_csv)
if pareto_df.empty:
    raise ValueError(f'Pareto CSV is empty: {pareto_csv}')

print('Using Pareto file:', pareto_csv)
print('Total Pareto points:', len(pareto_df))
display(pareto_df[['trial_number', 'accuracy', 'inference_ms', 'model_params']].head(10))

In [ ]:
import numpy as np

# Choose one strategy: 'balanced', 'best_accuracy', 'fastest', 'smallest', 'trial_number'
SELECTION_MODE = 'balanced'
TRIAL_NUMBER = None  # set an int if SELECTION_MODE='trial_number'

def pick_pareto_row(df: pd.DataFrame, mode: str, trial_number=None) -> pd.Series:
    mode = mode.lower()

    if mode == 'trial_number':
        if trial_number is None:
            raise ValueError('Set TRIAL_NUMBER when using mode=trial_number')
        rows = df[df['trial_number'] == trial_number]
        if rows.empty:
            raise ValueError(f'Trial number {trial_number} not found in Pareto CSV')
        return rows.iloc[0]

    if mode == 'best_accuracy':
        return df.sort_values('accuracy', ascending=False).iloc[0]

    if mode == 'fastest':
        return df.sort_values('inference_ms', ascending=True).iloc[0]

    if mode == 'smallest':
        return df.sort_values('model_params', ascending=True).iloc[0]

    if mode == 'balanced':
        tmp = df[['accuracy', 'inference_ms', 'model_params']].copy()
        tmp['accuracy'] = 1.0 - (tmp['accuracy'] - tmp['accuracy'].min()) / (tmp['accuracy'].max() - tmp['accuracy'].min() + 1e-12)
        tmp['inference_ms'] = (tmp['inference_ms'] - tmp['inference_ms'].min()) / (tmp['inference_ms'].max() - tmp['inference_ms'].min() + 1e-12)
        tmp['model_params'] = (tmp['model_params'] - tmp['model_params'].min()) / (tmp['model_params'].max() - tmp['model_params'].min() + 1e-12)
        score = tmp.mean(axis=1)
        return df.iloc[int(score.argmin())]

    raise ValueError(f'Unknown SELECTION_MODE: {mode}')

selected = pick_pareto_row(pareto_df, SELECTION_MODE, TRIAL_NUMBER)
print('Selected trial_number:', int(selected['trial_number']))
display(selected.to_frame().T)

In [ ]:
import torch

from src.config import load_config
from src.data import build_dataloaders
from src.model import DynamicCNN
from src.train_eval import train_one_model
from src.utils import count_parameters, select_device, set_seed

cfg = load_config(str(PROJECT_ROOT / 'configs' / 'default.yaml')).data
set_seed(int(cfg['seed']))

n_conv_layers = int(selected['n_conv_layers'])
n_fc_layers = int(selected['n_fc_layers'])
conv_channels = [int(selected[f'conv_channels_l{i+1}']) for i in range(n_conv_layers)]
hidden_units = [int(selected[f'hidden_units_l{i+1}']) for i in range(n_fc_layers)]

trial_cfg = {
    'input_resolution': int(selected['input_resolution']),
    'batch_size': int(selected['batch_size']),
    'epochs': int(selected['epochs']),
    'learning_rate': float(selected['learning_rate']),
    'optimizer': str(selected['optimizer']),
    'dropout': float(selected['dropout']),
    'n_conv_layers': n_conv_layers,
    'conv_channels': conv_channels,
    'n_fc_layers': n_fc_layers,
    'hidden_units': hidden_units,
}

device = select_device()
print('Device:', device)
if device.type == 'cuda':
    print('CUDA devices:', torch.cuda.device_count())

loaders = build_dataloaders(
    data_dir=str(PROJECT_ROOT / cfg['dataset']['data_dir']),
    input_resolution=trial_cfg['input_resolution'],
    batch_size=trial_cfg['batch_size'],
    train_subset=int(cfg['dataset']['train_subset']),
    val_subset=int(cfg['dataset']['val_subset']),
    test_subset=int(cfg['dataset']['test_subset']),
    num_workers=int(cfg['dataset']['num_workers']),
    seed=int(cfg['seed']),
)

model = DynamicCNN(
    input_resolution=trial_cfg['input_resolution'],
    n_conv_layers=trial_cfg['n_conv_layers'],
    conv_channels=trial_cfg['conv_channels'],
    n_fc_layers=trial_cfg['n_fc_layers'],
    hidden_units=trial_cfg['hidden_units'],
    dropout=trial_cfg['dropout'],
    num_classes=10,
)

if device.type == 'cuda' and torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

print('Model parameters:', count_parameters(model))
print('Training config:', trial_cfg)

In [ ]:
from pathlib import Path

best_val_acc = train_one_model(
    model=model,
    train_loader=loaders.train_loader,
    val_loader=loaders.val_loader,
    optimizer_name=trial_cfg['optimizer'],
    learning_rate=trial_cfg['learning_rate'],
    epochs=trial_cfg['epochs'],
    device=device,
)

print(f'Best validation accuracy during training: {best_val_acc:.4f}')

ckpt_dir = PROJECT_ROOT / 'outputs'
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / f'retrained_trial_{int(selected["trial_number"]):04d}.pt'
state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
torch.save(state_dict, ckpt_path)
print('Saved checkpoint:', ckpt_path)

In [ ]:
from src.train_eval import evaluate_accuracy, full_evaluation

noise_std = float(cfg['objectives']['gaussian_noise_std'])
val_acc = evaluate_accuracy(model, loaders.val_loader, device)
test_eval = full_evaluation(model, loaders.test_loader, noise_std=noise_std, device=device)

print('Validation Accuracy (non-training):', f'{val_acc:.4f}')
print('Test Accuracy (non-training):', f'{test_eval.clean_accuracy:.4f}')
print('Noisy Test Accuracy:', f'{test_eval.noisy_accuracy:.4f}')
print('Inference Time (ms/sample):', f'{test_eval.inference_ms_per_sample:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for images, labels in loaders.test_loader:
        images = images.to(device)
        logits = model(images)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_targets.extend(labels.numpy().tolist())

print('Classification report on test data:')
print(classification_report(all_targets, all_preds, target_names=class_names, digits=4))

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix (Test Data)')
plt.colorbar()
ticks = np.arange(len(class_names))
plt.xticks(ticks, class_names, rotation=45, ha='right')
plt.yticks(ticks, class_names)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize a few sample predictions from non-training (test) data
batch_images, batch_labels = next(iter(loaders.test_loader))
batch_images_device = batch_images.to(device)

with torch.no_grad():
    logits = model(batch_images_device)
    batch_preds = torch.argmax(logits, dim=1).cpu().numpy()

batch_images_np = batch_images.cpu().numpy()
batch_labels_np = batch_labels.numpy()

n_show = min(12, len(batch_images_np))
plt.figure(figsize=(14, 6))
for i in range(n_show):
    plt.subplot(3, 4, i + 1)
    img = (batch_images_np[i, 0] * 0.5) + 0.5
    plt.imshow(img, cmap='gray')
    true_name = class_names[batch_labels_np[i]]
    pred_name = class_names[batch_preds[i]]
    color = 'green' if batch_labels_np[i] == batch_preds[i] else 'red'
    plt.title(f'T:{true_name}\nP:{pred_name}', color=color, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()